In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.model_selection import TimeSeriesSplit, cross_validate, GridSearchCV
from sklearn.metrics import( 
    classification_report,
    confusion_matrix,
    f1_score, 
    accuracy_score,
    precision_score,
    recall_score,
    ConfusionMatrixDisplay
)
from catboost import CatBoostClassifier

In [2]:
# CSV file path
DF_HEALTH_PATH = "new-data-security-incident-trends-health-sector.csv"
# Target feature
TARGET_COL = "decision_taken"
# Multilabel columns contain a comma separated list of values and need multi label encoding
MULTILABEL_COLS = ["data_subject_type", "data_type"]
# Categorical columns are single value
CATEGORICAL_COLS = ["incident_category", "incident_type", "no_data_subjects_affected", "time_taken_to_report"]
# All feature columns
FEATURE_COLS = MULTILABEL_COLS + CATEGORICAL_COLS
# Class Labels
CLASS_LABELS = ["Informal Action Taken", "Investigation Pursued", "No Further Action"]
# Random seed for reproducibility
RANDOM_STATE = 42
# Cross validation splits
CV_SPLITS = 5

In [3]:
# Load the preprocessed dataset
df = pd.read_csv(DF_HEALTH_PATH)

# Initial check for null values and data types - columns are int64 and object, no nulls
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9250 entries, 0 to 9249
Data columns (total 9 columns):
 #   Column                     Non-Null Count  Dtype 
---  ------                     --------------  ----- 
 0   year                       9250 non-null   int64 
 1   quarter                    9250 non-null   object
 2   data_subject_type          9250 non-null   object
 3   data_type                  9250 non-null   object
 4   decision_taken             9250 non-null   object
 5   incident_category          9250 non-null   object
 6   incident_type              9250 non-null   object
 7   no_data_subjects_affected  9250 non-null   object
 8   time_taken_to_report       9250 non-null   object
dtypes: int64(1), object(8)
memory usage: 650.5+ KB


In [ ]:
# Identical temporal split to models.ipynb: train on 2021-2024, test on 2025
def is_test_split(row): return row["year"] == 2025

df["is_test"] = df.apply(is_test_split, axis=1)

df_train = df[df["is_test"] == False].reset_index(drop=True)
df_test = df[df["is_test"] == True].reset_index(drop=True)

df = df.drop(columns=["is_test"])
df_train = df_train.drop(columns=["is_test"])
df_test = df_test.drop(columns=["is_test"])

print(f"Training rows : {len(df_train)}")
print(f"Test rows : {len(df_test)}")
print(f"\nTraining periods:")
print(df_train[["year", "quarter"]].drop_duplicates().sort_values(["year", "quarter"]).to_string(index=False))
print(f"\nTest periods:")
print(df_test[["year", "quarter"]].drop_duplicates().sort_values(["year", "quarter"]).to_string(index=False))

Training rows : 7312
Test rows : 1938

Training periods:
 year quarter
 2021   Qtr 2
 2021   Qtr 3
 2021   Qtr 4
 2022   Qtr 1
 2022   Qtr 2
 2022   Qtr 3
 2022   Qtr 4
 2023   Qtr 1
 2023   Qtr 2
 2023   Qtr 3
 2023   Qtr 4
 2024   Qtr 1
 2024   Qtr 2
 2024   Qtr 3
 2024   Qtr 4

Test periods:
 year quarter
 2025   Qtr 1
 2025   Qtr 2
 2025   Qtr 3
 2025   Qtr 4


In [ ]:
# Raw categorical columns, keep as strings
cat_train = df_train[CATEGORICAL_COLS].reset_index(drop=True)
cat_test  = df_test[CATEGORICAL_COLS].reset_index(drop=True)

# MLB encoded columns
mlb_train = pd.concat(mlb_train_frames, axis=1).reset_index(drop=True)
mlb_test  = pd.concat(mlb_test_frames,  axis=1).reset_index(drop=True)

# Combine into final feature matrices
X_train = pd.concat([cat_train, mlb_train], axis=1)
X_test  = pd.concat([cat_test,  mlb_test],  axis=1)

y_train = df_train[TARGET_COL]
y_test  = df_test[TARGET_COL]

# Identify the column indices of categorical features for CatBoost
# CatBoost needs integer indices, not column names, when using DataFrames
cat_feature_indices = [X_train.columns.get_loc(col) for col in CATEGORICAL_COLS]

print(f"X_train shape : {X_train.shape}")
print(f"X_test shape : {X_test.shape}")
print(f"Categorical indices : {cat_feature_indices}")
print(f"Categorical columns : {CATEGORICAL_COLS}")
print(f"\nSample of X_train (first 3 rows):")
print(X_train.head(3).to_string())

assert X_train.isnull().sum().sum() == 0, "Nulls found in X_train"
assert X_test.isnull().sum().sum() == 0, "Nulls found in X_test"
print("\nNo nulls in feature matrices")

In [ ]:
# identical to models.ipynb for consistency
def evaluate_model(model_name, y_true, y_pred):
    print(f"{model_name}: Evaluation")

    print("Classification Report:")
    print(classification_report(y_true, y_pred, labels=CLASS_LABELS, zero_division=0))

    acc = accuracy_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred, average="macro", zero_division=0)
    prec = precision_score(y_true, y_pred, average="macro", zero_division=0)
    rec = recall_score(y_true, y_pred, average="macro", zero_division=0)

    print(f"Accuracy : {acc:.4f}")
    print(f"Macro F1 : {f1:.4f}")
    print(f"Macro Precision : {prec:.4f}")
    print(f"Macro Recall : {rec:.4f}")

    cm = confusion_matrix(y_true, y_pred, labels=CLASS_LABELS)
    fig, ax = plt.subplots(figsize=(7, 5))
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=CLASS_LABELS)
    disp.plot(cmap="Blues", ax=ax, colorbar=False)
    ax.set_title(f"Confusion Matrix — {model_name}", fontsize=13, fontweight="bold", pad=12)
    plt.xticks(rotation=20, ha="right")
    plt.tight_layout()
    plt.show()

    return {
        "model" : model_name,
        "accuracy" : round(acc,  4),
        "macro_f1" : round(f1,   4),
        "precision": round(prec, 4),
        "recall" : round(rec,  4),
    }